In [71]:
using DataFrames, Distributions, CSV, Turing, Optim, StatsBase, StatsFuns
include("measures.jl")

# Read the vitamin D dataset
data = CSV.read("../../data/vitd/nhanes.csv", DataFrame);
data[!, :"RIAGENDR"] = data[!, :"RIAGENDR"] .- 1;

# Look at the first few rows of the data
first(data, 5)

Row,BMXBMI,RIAGENDR,RIDAGEYR,LBXVIDMS,INDFMMPI,DPQ020,SMQ040
,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,27.8,0.0,62.0,76.1,4.14,0.0,3.0
2,30.8,0.0,53.0,56.5,0.0,0.0,1.0
3,28.8,0.0,78.0,87.5,1.81,0.0,3.0
4,28.0,0.0,22.0,47.2,2.98,0.0,2.0
5,27.6,0.0,46.0,44.5,1.73,0.0,1.0


In [72]:
#= Convert categorical data to integers =#

data[!, :"RIAGENDR"] = convert(Array{Int}, data[!, :"RIAGENDR"])
data[!, :"SMQ040"] = convert(Array{Int}, data[!, :"SMQ040"]);

In [63]:
# Get variables
bmi = data[!, :"BMXBMI"]
gender = data[!, :"RIAGENDR"]
age = data[!, :"RIDAGEYR"]
vitd = data[!, :"LBXVIDMS"]
poverty = data[!, :"INDFMMPI"]
depression = data[!, :"DPQ020"]
smoking = data[!, :"SMQ040"];

In [64]:
# Turing Model NO BIDRECTIONS

@model function no_depression_model(bmi, gender, age, vitd, poverty, depression, smoking)
    n = length(vitd)

    # Define the priors
    α_bmi ~ Normal(0, 1)
    β_gender_bmi ~ Normal(0, 1)
    β_age_bmi ~ Normal(0, 1)

    α_vitd ~ Normal(0, 1)
    β_bmi_vitd ~ Normal(0, 1)
    β_age_vitd ~ Normal(0, 1)
    β_poverty_vitd ~ Normal(0, 1)
    β_gender_vitd ~ Normal(0, 1)
    β_smoking_vitd ~ Normal(0, 1)

    # Define the linear models
    for i in 1:n
        gender[i] ~ Bernoulli(0.4)
        age_dist1 = Normal(36, 10)
        age_dist2 = Normal(62, 10)
        
        # Latent variable for depression
        depression[i] ~ LogNormal(0.02, 0.6)

        # Round and clamp the depression levels
        depression[i] = clamp(round(Int, depression[i]), 1, 4)
        
        age[i] ~ MixtureModel([age_dist1, age_dist2], [0.4, 0.6])
        smoking[i] ~ Categorical([0.35, 0.12, 0.53])
    
        bmi[i] ~ LogNormal(α_bmi + gender[i] * β_gender_bmi + age[i] * β_age_bmi, 0.22)
        poverty_dist1 = Normal(1, 2)
        poverty_dist2 = Normal(5, 0.001)
        poverty[i] ~ MixtureModel([poverty_dist1, poverty_dist2], [0.5, 0.5])

        vitd[i] ~ LogNormal(α_vitd + bmi[i] * β_bmi_vitd + age[i] * β_age_vitd + poverty[i] * β_poverty_vitd + gender[i] * β_gender_vitd + smoking[i] * β_smoking_vitd, 0.42)
    end
end

no_depression_model (generic function with 2 methods)

In [65]:
cont_sampler = HMC(0.01, 10, :bmi, :age, :poverty, :vitd)
disc_sampler = PG(10, :gender, :depression, :smoking)
sampler = Gibbs(cont_sampler, disc_sampler)

Gibbs{(:bmi, :age, :poverty, :vitd, :gender, :depression, :smoking), Tuple{HMC{Turing.Essential.ForwardDiffAD{0}, (:bmi, :age, :poverty, :vitd), AdvancedHMC.UnitEuclideanMetric}, PG{(:gender, :depression, :smoking), AdvancedPS.ResampleWithESSThreshold{typeof(AdvancedPS.resample_systematic), Float64}}}}((HMC{Turing.Essential.ForwardDiffAD{0}, (:bmi, :age, :poverty, :vitd), AdvancedHMC.UnitEuclideanMetric}(0.01, 10), PG{(:gender, :depression, :smoking), AdvancedPS.ResampleWithESSThreshold{typeof(AdvancedPS.resample_systematic), Float64}}(10, AdvancedPS.ResampleWithESSThreshold{typeof(AdvancedPS.resample_systematic), Float64}(AdvancedPS.resample_systematic, 0.5))))

In [66]:
model_optim = no_depression_model(bmi, gender, age, vitd, poverty, depression, smoking)

DynamicPPL.Model{typeof(no_depression_model), (:bmi, :gender, :age, :vitd, :poverty, :depression, :smoking), (), (), Tuple{SentinelArrays.ChainedVector{Float64, Vector{Float64}}, Vector{Int64}, SentinelArrays.ChainedVector{Float64, Vector{Float64}}, SentinelArrays.ChainedVector{Float64, Vector{Float64}}, SentinelArrays.ChainedVector{Float64, Vector{Float64}}, SentinelArrays.ChainedVector{Float64, Vector{Float64}}, Vector{Int64}}, Tuple{}, DynamicPPL.DefaultContext}(no_depression_model, (bmi = [27.8, 30.8, 28.8, 28.0, 27.6, 24.1, 26.6, 30.3, 35.9, 31.0  …  32.8, 18.8, 32.3, 24.0, 25.1, 53.9, 37.6, 23.4, 32.0, 21.5], gender = [0, 0, 0, 0, 0, 0, 1, 1, 1, 1  …  0, 0, 0, 0, 0, 1, 1, 1, 0, 1], age = [62.0, 53.0, 78.0, 22.0, 46.0, 45.0, 30.0, 69.0, 60.0, 69.0  …  48.0, 63.0, 72.0, 38.0, 62.0, 73.0, 63.0, 72.0, 53.0, 76.0], vitd = [76.1, 56.5, 87.5, 47.2, 44.5, 74.5, 46.3, 103.0, 14.5, 96.5  …  82.6, 39.0, 58.1, 57.8, 73.1, 80.4, 85.2, 91.0, 52.7, 65.5], poverty = [4.14, 0.0, 1.81, 2.98, 1.73,

In [67]:
map_estimate = optimize(model_optim, MAP(), LBFGS(), Optim.Options(iterations=1000000, show_trace=true, show_every=100))

Iter     Function value   Gradient norm 
     0     2.281661e+07     3.256055e+07
 * time: 0.0002300739288330078


ModeResult with maximized lp of -26094.12
9-element Named Vector{Float64}
A               │ 
────────────────┼────────────
:α_bmi          │     3.32873
:β_gender_bmi   │   0.0453978
:β_age_bmi      │ 0.000489447
:α_vitd         │     3.94651
:β_bmi_vitd     │  -0.0109559
:β_age_vitd     │   0.0048478
:β_poverty_vitd │   0.0355752
:β_gender_vitd  │   0.0640624
:β_smoking_vitd │   0.0521531

In [68]:
# Turing Model NO BIDRECTIONS

@model function depression_model(bmi, gender, age, vitd, poverty, depression, smoking)
    n = length(vitd)

    # Define the priors
    α_depression ~ Normal(0, 1.0)

    β_vitd_depression ~ Normal(0, 1.0)

    β_age_depression ~ Normal(0, 1.0)

    β_gender_depression ~ Normal(0, 1.0)

    β_smoking_depression ~ Normal(0, 1.0)

    # Define the linear models
    
    for i in 1:n
        gender[i] ~ Bernoulli(0.4)
        age_dist1 = Normal(36, 10)
        age_dist2 = Normal(62, 10)
        age[i] ~ MixtureModel([age_dist1, age_dist2], [0.4, 0.6])
        smoking[i] ~ Categorical([0.35, 0.12, 0.53])
    
        bmi[i] ~ LogNormal(3.28276 + gender[i] * 0.045803 + age[i] * 0.000489506, 0.22)
        poverty_dist1 = Normal(2.07895, 2)
        poverty_dist2 = Normal(1.05, 0.001)
        poverty[i] ~ MixtureModel([poverty_dist1, poverty_dist2], [0.5, 0.5])

       # Latent variable for depression
       depression[i] ~ LogNormal(α_depression + β_vitd_depression * vitd[i] + β_age_depression * age[i]
       + β_gender_depression * gender[i] + β_smoking_depression * smoking[i], 0.6)  # Center around 2.5 to allow values to range from 1 to 4

       # Round and clamp the depression levels
       depression[i] = clamp(round(Int, depression[i]), 1, 4)

        vitd[i] ~ LogNormal(3.8805 + bmi[i] * -0.0109757 + age[i] * 0.00484755 + poverty[i] * 0.0355696 + gender[i] * 0.0658405 + smoking[i] * 0.0521878, 0.42)
    end
end

depression_model (generic function with 2 methods)

In [69]:
model_optim = depression_model(bmi, gender, age, vitd, poverty, sedentary, depression, smoking)

UndefVarError: UndefVarError: `sedentary` not defined

In [70]:
map_estimate = optimize(model_optim, MAP(), LBFGS(), Optim.Options(iterations=1000000, show_trace=true, show_every=100))

Iter     Function value   Gradient norm 
     0     5.251802e+07     8.056814e+07
 * time: 6.198883056640625e-5


ModeResult with maximized lp of -26094.12
9-element Named Vector{Float64}
A               │ 
────────────────┼────────────
:α_bmi          │     3.32873
:β_gender_bmi   │   0.0453978
:β_age_bmi      │ 0.000489447
:α_vitd         │     3.94651
:β_bmi_vitd     │  -0.0109559
:β_age_vitd     │   0.0048478
:β_poverty_vitd │   0.0355752
:β_gender_vitd  │   0.0640624
:β_smoking_vitd │   0.0521531